# 데이터팀 특징 추출 노트북

- 작성: 양윤서 | 기준 문서: 데이터팀 공통 규칙
- **사용법**: 셀 2(설정)에서 `DATASET`과 `TRIM`만 바꾸고, **런타임 → 모두 실행**을 누른다.
- 한 번 실행하면 CSV 1개가 `feature` 폴더에 저장된다. mon·unmon × trim 0·10·20 = 6번 실행한다.
- mon·trim0으로 실행할 때는 누적 흐름 100개 지점 비교용 `cumul100_mon_trim0.csv`도 함께 저장된다.

| 구간 | 셀 | 내용 |
|---|---|---|
| 1부 | 1~7 | 드라이브 연결, 설정, 데이터 로드, 전처리, 전처리 검증 |
| 2부 | 8~14 | 그룹 1~6 특징 계산 함수 |
| 3부 | 15~18 | 예시 기록 확인, 전체 추출, 결과 검증, CSV 저장 |


## [셀 1] 구글 드라이브 연결  (Colab에서만 실행)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## [셀 2] 설정  ← 실행하는 사람은 여기만 바꾸면 됨

In [ ]:
DATA_DIR = '/content/drive/MyDrive/MLB-Project/data'     # pkl 파일이 있는 폴더
OUT_DIR  = '/content/drive/MyDrive/MLB-Project/feature'  # 결과 CSV를 저장할 폴더

DATASET = 'mon'   # 'mon' / 'unmon10k' / 'unmon3k'
TRIM    = 0       # 0(원본) / 10(앞 10개 제거) / 20(앞 20개 제거)


## [셀 3] 라이브러리 불러오기

In [ ]:
import os, pickle
import numpy as np
import pandas as pd

FILE_NAMES = {
    'mon':      'mon_standard.pkl',
    'unmon10k': 'unmon_standard10.pkl',
    'unmon3k':  'unmon_standard10_3000.pkl',
}


## [셀 4] 데이터 로드: 기록마다 ID·라벨 붙이기 (공통 규칙 2번)

In [ ]:
def load_records(dataset):
    """pkl 파일을 읽어서 기록 목록을 만든다.
    기록 하나 = {record_id, dataset, page_id, sample_idx, label, raw}"""
    path = os.path.join(DATA_DIR, FILE_NAMES[dataset])
    with open(path, 'rb') as f:
        data = pickle.load(f)

    records = []
    if dataset == 'mon':
        # mon: {페이지 번호(0~949): [접속 기록 20개]}
        for page_id in range(950):
            for sample_idx, raw in enumerate(data[page_id]):
                records.append({
                    'record_id': f'mon_p{page_id:03d}_s{sample_idx:02d}',
                    'dataset': 'mon',
                    'page_id': page_id,
                    'sample_idx': sample_idx,
                    'label': page_id // 10,          # 사이트 번호 0~94
                    'raw': raw,
                })
    else:
        # unmon: [기록, 기록, ...] 페이지 구분 없음
        for i, raw in enumerate(data):
            records.append({
                'record_id': f'{dataset}_{i:05d}',
                'dataset': dataset,
                'page_id': -1,
                'sample_idx': i,
                'label': -1,
                'raw': raw,
            })
    return records


## [셀 5] 전처리: 시각·방향 분리 + 앞부분 제거 (공통 규칙 3, 4번)

In [ ]:
def preprocess(raw, trim=0):
    """원본 숫자 목록 → (시각 배열, 방향 배열)
    - 방향: 양수면 송신(+1), 0 이하면 수신(-1)
    - 시각: 절댓값(초)
    - trim > 0이면 앞 trim개를 지우고, 남은 첫 시각이 0이 되도록 다시 맞춘다"""
    raw = np.asarray(raw, dtype=np.float64)
    dirs = np.where(raw > 0, 1, -1)
    times = np.abs(raw)
    if trim > 0:
        dirs = dirs[trim:]
        times = times[trim:]
        times = times - times[0]
    return times, dirs


## [셀 6] 전처리 검증 (공통 규칙 9번)

In [ ]:
def validate(records, trim=0):
    lengths, backward, zero_dur, no_out = [], 0, 0, 0
    for r in records:
        times, dirs = preprocess(r['raw'], trim)
        lengths.append(len(times))
        if np.any(np.diff(times) < 0):
            backward += 1
        if times[-1] - times[0] == 0:
            zero_dur += 1
        if not np.any(dirs == 1):
            no_out += 1
    lengths = np.array(lengths)
    print(f'[검증] 데이터: {DATASET}, 제거 조건: trim{trim}')
    print(f'  기록 수: {len(records):,}개')
    print(f'  기록 길이: 최소 {lengths.min():,} / 중앙값 {int(np.median(lengths)):,} / 최대 {lengths.max():,}')
    print(f'  시각이 역행하는 기록: {backward}개')
    print(f'  지속 시간이 0인 기록: {zero_dur}개')
    print(f'  송신 패킷이 없는 기록: {no_out}개')


## [셀 7] 실행: 불러오기 → 검증 → 예시 기록 하나 확인

In [ ]:
records = load_records(DATASET)
validate(records, TRIM)

ex = records[0]
t, d = preprocess(ex['raw'], TRIM)
print('\n[예시 기록]', ex['record_id'], '| 라벨:', ex['label'])
print('  원본 앞 10개:', [round(x, 2) for x in ex['raw'][:10]])
print('  시각 앞 10개:', np.round(t[:10], 2).tolist())
print('  방향 앞 10개:', d[:10].tolist())


## [셀 8] 추가 설정

In [ ]:
INCLUDE_PARTIAL_CHUNK = False   # 그룹 2: 마지막 불완전 구간 포함 여부 (공통 규칙 7-(2), 기본: 제외)
CUMUL_POINTS = 50               # 그룹 6: 누적 흐름 지점 수
MAKE_CUMUL100 = True            # mon·trim0일 때 100개 지점 비교용 파일도 만들지 여부


## [셀 9] 그룹 1. 패킷 개수·비율 (3개)

In [ ]:
def group1(times, dirs):
    n = len(dirs)
    n_out = int(np.sum(dirs == 1))
    return {
        'g1_n_total': n,
        'g1_n_out': n_out,
        'g1_in_ratio': (n - n_out) / n,
    }


## [셀 10] 그룹 2. 패킷 순서·위치 (17개)

In [ ]:
def _stats5(x):
    """평균, 표준편차, 최대, 최소, 중앙값"""
    x = np.asarray(x, dtype=np.float64)
    return [x.mean(), x.std(), x.max(), x.min(), np.median(x)]

def group2(times, dirs):
    n = len(dirs)
    is_out = (dirs == 1)
    out_pos = np.flatnonzero(is_out)          # 송신 패킷의 위치(0부터)
    f = {
        'g2_out_pos_mean': out_pos.mean() if len(out_pos) else 0.0,
        'g2_out_pos_std':  out_pos.std()  if len(out_pos) else 0.0,
    }
    for size in (20, 50, 100):
        k = n // size                          # 완전한 구간 개수
        if k == 0:
            counts = [is_out.sum()]            # 완전한 구간이 없으면 기록 전체를 한 구간으로
        else:
            counts = list(is_out[:k * size].reshape(k, size).sum(axis=1))
            if INCLUDE_PARTIAL_CHUNK and n % size:
                counts.append(is_out[k * size:].sum())
        for name, v in zip(['mean', 'std', 'max', 'min', 'median'], _stats5(counts)):
            f[f'g2_chunk{size}_{name}'] = v
    return f


## [셀 11] 그룹 3. 연속 묶음(버스트) (6개)

In [ ]:
def group3(times, dirs):
    n = len(dirs)
    starts = np.r_[0, np.flatnonzero(np.diff(dirs)) + 1]   # 방향이 바뀌는 위치 = 묶음 시작
    lengths = np.diff(np.r_[starts, n])                    # 묶음 길이
    bdir = dirs[starts]                                    # 묶음 방향
    out_len, in_len = lengths[bdir == 1], lengths[bdir == -1]
    return {
        'g3_n_out_bursts':   int(len(out_len)),
        'g3_bursts_len1_5':  int(np.sum(lengths <= 5)),
        'g3_bursts_len6_20': int(np.sum((lengths >= 6) & (lengths <= 20))),
        'g3_bursts_len21p':  int(np.sum(lengths >= 21)),
        'g3_max_out_burst':  int(out_len.max()) if len(out_len) else 0,
        'g3_max_in_burst':   int(in_len.max())  if len(in_len)  else 0,
    }


## [셀 12] 그룹 4. 접속 초반·후반의 방향 구성 (3개)

In [ ]:
def group4(times, dirs):
    is_out = (dirs == 1)
    return {   # 기록이 짧으면 있는 패킷만으로 계산
        'g4_first30_out_ratio':  is_out[:30].mean(),
        'g4_first100_out_ratio': is_out[:100].mean(),
        'g4_last30_out_ratio':   is_out[-30:].mean(),
    }


## [셀 13] 그룹 5. 시간·통신 속도 (31개)

In [ ]:
def _iat_stats(x):
    """평균, 표준편차, 최대, 25%, 50%, 75%. 간격이 없으면 모두 0"""
    if len(x) == 0:
        return [0.0] * 6
    q25, q50, q75 = np.percentile(x, [25, 50, 75])
    return [x.mean(), x.std(), x.max(), q25, q50, q75]

def group5(times, dirs):
    n = len(times)
    t = times - times[0]
    duration = t[-1]
    f = {'g5_duration': duration}

    # 1초 구간별 패킷 수 (패킷이 없는 1초도 0으로 포함)
    if duration == 0:
        pps = np.array([n])
    else:
        pps = np.bincount(np.floor(t).astype(int), minlength=int(np.floor(duration)) + 1)
    for name, v in zip(['mean', 'std', 'max', 'min', 'median'], _stats5(pps)):
        f[f'g5_pps_{name}'] = v

    # 상대 도착 시점: 전체 중 q%번째 패킷의 시각 ÷ 지속 시간
    for q in (25, 50, 75):
        idx = int(np.ceil(q / 100 * n)) - 1
        f[f'g5_rel_t{q}'] = t[idx] / duration if duration > 0 else 0.0

    # 시간 간격: 전체 / 송신끼리 / 수신끼리
    iat_all = np.diff(t)
    groups = {'all': iat_all, 'out': np.diff(t[dirs == 1]), 'in': np.diff(t[dirs == -1])}
    for g, x in groups.items():
        for name, v in zip(['mean', 'std', 'max', 'q25', 'q50', 'q75'], _iat_stats(x)):
            f[f'g5_iat_{g}_{name}'] = v

    # 간격 0 비율, 0보다 큰 간격 통계
    f['g5_iat_zero_ratio'] = np.mean(iat_all == 0) if len(iat_all) else 0.0
    pos = iat_all[iat_all > 0]
    f['g5_iat_pos_mean'] = pos.mean() if len(pos) else 0.0
    f['g5_iat_pos_std']  = pos.std()  if len(pos) else 0.0
    f['g5_iat_pos_max']  = pos.max()  if len(pos) else 0.0
    return f


## [셀 14] 그룹 6. 전체 통신의 누적 흐름 (50개)

In [ ]:
def cumul_points(dirs, points):
    """송신 +1, 수신 -1 누적값을 처음~끝 같은 간격 points개 지점에서 보간"""
    cum = np.cumsum(dirs)
    x = np.linspace(0, len(dirs) - 1, points)
    return np.interp(x, np.arange(len(dirs)), cum)

def group6(times, dirs):
    vals = cumul_points(dirs, CUMUL_POINTS)
    return {f'g6_cumul_{i:02d}': v for i, v in enumerate(vals)}


## [셀 15] 예시 기록으로 계산 확인 (노션 정리본의 12개 예시와 비교)

In [ ]:
def extract_features(times, dirs):
    f = {}
    for g in (group1, group2, group3, group4, group5, group6):
        f.update(g(times, dirs))
    return f

example = [0.0, -0.10, 0.10, -0.20, -0.20, -0.30, 0.50, -0.60, -0.60, -0.60, -1.10, -1.20]
et, ed = preprocess(example, 0)
ef = extract_features(et, ed)
print('특징 개수:', len(ef))
for k in ['g1_n_total', 'g1_n_out', 'g1_in_ratio', 'g2_out_pos_mean', 'g2_out_pos_std',
          'g3_n_out_bursts', 'g3_bursts_len1_5', 'g3_max_out_burst', 'g3_max_in_burst',
          'g4_first30_out_ratio', 'g5_duration', 'g5_pps_mean', 'g5_pps_max',
          'g5_rel_t25', 'g5_rel_t50', 'g5_rel_t75', 'g5_iat_zero_ratio', 'g6_cumul_00', 'g6_cumul_49']:
    print(f'  {k:22s} = {round(float(ef[k]), 4)}')


## [셀 16] 전체 기록 특징 추출

In [ ]:
import time
start = time.time()
rows, cumul100 = [], []
for i, r in enumerate(records):
    times, dirs = preprocess(r['raw'], TRIM)
    row = {'record_id': r['record_id'], 'dataset': r['dataset'], 'page_id': r['page_id'],
           'sample_idx': r['sample_idx'], 'label': r['label'], 'trim': TRIM}
    row.update(extract_features(times, dirs))
    rows.append(row)
    if MAKE_CUMUL100 and DATASET == 'mon' and TRIM == 0:
        c = cumul_points(dirs, 100)
        cumul100.append({'record_id': r['record_id'], **{f'g6_cumul100_{j:03d}': v for j, v in enumerate(c)}})
    if (i + 1) % 2000 == 0:
        print(f'  {i + 1:,} / {len(records):,} 완료 ({time.time() - start:.0f}초)')
df = pd.DataFrame(rows)
print(f'추출 완료: {df.shape[0]:,}행 × {df.shape[1]}열 ({time.time() - start:.0f}초)')


## [셀 17] 결과 검증 (공통 규칙 9번)

In [ ]:
feat_cols = [c for c in df.columns if c.startswith('g')]
values = df[feat_cols].to_numpy(dtype=np.float64)
print('[결과 검증]')
print(f'  기록 수: {len(df):,}개')
print(f'  전체 열 수: {df.shape[1]}개 (정보 6 + 특징 {len(feat_cols)})')
print(f'  빈칸(NaN): {int(np.isnan(values).sum())}개')
print(f'  무한대 값: {int(np.isinf(values).sum())}개')
ratio_cols = [c for c in feat_cols if c.endswith('_ratio') or '_rel_t' in c]
out_of_range = ((df[ratio_cols] < 0) | (df[ratio_cols] > 1)).any(axis=1).sum()
print(f'  비율 특징이 0~1 범위를 벗어난 기록: {int(out_of_range)}개')
print('[대체값(0)을 사용한 기록 수]')
print(f'  송신 패킷이 2개 미만 (송신 간격 계산 불가): {int((df["g1_n_out"] < 2).sum())}개')
print(f'  수신 패킷이 2개 미만 (수신 간격 계산 불가): {int(((df["g1_n_total"] - df["g1_n_out"]) < 2).sum())}개')
print(f'  지속 시간이 0: {int((df["g5_duration"] == 0).sum())}개')
print(f'  0보다 큰 간격이 없음: {int((df["g5_iat_pos_max"] == 0).sum())}개')


## [셀 18] CSV 저장

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, f'features_{DATASET}_trim{TRIM}.csv')
df.to_csv(out_path, index=False, float_format='%.6g', encoding='utf-8')
print('저장 완료:', out_path, f'({os.path.getsize(out_path) / 1e6:.1f} MB)')

if cumul100:
    c_path = os.path.join(OUT_DIR, 'cumul100_mon_trim0.csv')
    pd.DataFrame(cumul100).to_csv(c_path, index=False, float_format='%.6g', encoding='utf-8')
    print('저장 완료:', c_path, f'({os.path.getsize(c_path) / 1e6:.1f} MB)')

df.head()
